# Lecture 7 — Class Exercise
## Part-to-Whole: Hierarchical Visualization

> **Push to:** `week07/lecture07_exercise.ipynb`


In [8]:
import os
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        print(os.path.join(root, f))

/kaggle/input/datasets/prarthanap374/global/global_energy_mix.csv


In [9]:
import pandas as pd
import plotly.express as px
import numpy as np

# Dataset: Global Energy Mix by Country and Source
df = pd.read_csv('/kaggle/input/datasets/prarthanap374/global/global_energy_mix.csv')

# Source type mapping — reuse from lecture
source_category = {
    'Coal': 'Fossil', 'Oil': 'Fossil', 'Natural Gas': 'Fossil',
    'Nuclear': 'Low-carbon', 'Hydro': 'Low-carbon',
    'Wind': 'Renewable', 'Solar': 'Renewable', 'Other Renewables': 'Renewable'
}
df['Source_Type'] = df['Source'].map(source_category)

print(f"Loaded: {len(df)} rows")
print(df.head(10))

Loaded: 103 rows
         Country         Region            Source  Share_pct     TWh  \
0  United States  North America              Coal         10  1015.0   
1  United States  North America               Oil         35  3220.0   
2  United States  North America       Natural Gas         34  3083.0   
3  United States  North America           Nuclear          9   798.0   
4  United States  North America             Hydro          3   339.0   
5  United States  North America              Wind          4   413.0   
6  United States  North America             Solar          3   325.0   
7  United States  North America  Other Renewables          2   229.0   
8          China           Asia              Coal         58  6929.1   
9          China           Asia               Oil         18  1620.0   

  Source_Type  
0      Fossil  
1      Fossil  
2      Fossil  
3  Low-carbon  
4  Low-carbon  
5   Renewable  
6   Renewable  
7   Renewable  
8      Fossil  
9      Fossil  


## Task 1 — Treemap: fossil fuel dependency by country

**What to build:** A treemap showing **fossil fuel TWh only**, broken down by Region → Country → Source (Coal / Oil / Natural Gas).

**Requirements:**
- Filter to fossil sources only before plotting
- Use `path=['Region', 'Country', 'Source']` for the hierarchy
- Colour encodes the fossil source type (Coal / Oil / Natural Gas) with a CVD-safe palette
- Show TWh values in labels — no percentages
- Grey out parent nodes (Region and Country level)
- Insight title naming which region or country is most fossil-dependent

> 💡 `df.loc[df['Source_Type'] == 'Fossil']`


In [10]:
# Task 1
# YOUR CODE HERE
df_fossil = df.loc[df['Source_Type'] == 'Fossil'].copy()

# CVD-safe (Okabe-Ito) colours; '(?)' = parent nodes, greyed out
colors = {
    'Coal': '#D55E00',
    'Oil': '#0072B2',
    'Natural Gas': '#E69F00',
    '(?)': '#D9D9D9',
}

# Insight values for the title
region_tot = df_fossil.groupby('Region')['TWh'].sum().sort_values(ascending=False)
country_tot = df_fossil.groupby('Country')['TWh'].sum().sort_values(ascending=False)
top_region, top_region_twh = region_tot.index[0], region_tot.iloc[0]
top_country, top_country_twh = country_tot.index[0], country_tot.iloc[0]

fig1 = px.treemap(
    df_fossil,
    path=['Region', 'Country', 'Source'],
    values='TWh',
    color='Source',
    color_discrete_map=colors,
)
fig1.update_traces(
    texttemplate='%{label}<br>%{value:,.0f} TWh',
    hovertemplate='%{label}<br>%{value:,.0f} TWh<extra></extra>',
)
fig1.update_layout(
    title=(f'{top_region} is the most fossil-dependent region '
           f'({top_region_twh:,.0f} TWh), led by {top_country} ({top_country_twh:,.0f} TWh)'),
    margin=dict(t=70, l=10, r=10, b=10),
)
fig1.show()

## Task 2 — Treemap vs bar: low-carbon energy by country

**What to build:** Build **both** a treemap and a horizontal bar chart showing total low-carbon TWh (Nuclear + Hydro) per country. Then answer the question in a markdown cell below.

**Requirements:**
- Filter to `Source_Type == 'Low-carbon'` and aggregate TWh by country
- Treemap: single-level `path=['All', 'Country']` with a dummy root node labelled `'Low-carbon'`
- Bar chart: sorted by TWh, horizontal orientation, CVD-safe colour
- Both charts show TWh values, not percentages
- Insight title on the bar chart naming the leading country

In [13]:
# Task 2 — charts
# YOUR CODE HERE
low = (df.loc[df['Source_Type'] == 'Low-carbon']
         .groupby('Country', as_index=False)['TWh'].sum()
         .sort_values('TWh', ascending=False))
low['All'] = 'Low-carbon'          # dummy root node

# --- Treemap ---
fig2a = px.treemap(
    low, path=['All', 'Country'], values='TWh',
    color='All',
    color_discrete_map={'Low-carbon': '#0072B2', '(?)': '#D9D9D9'},
)
fig2a.update_traces(
    texttemplate='%{label}<br>%{value:,.0f} TWh',
    hovertemplate='%{label}<br>%{value:,.0f} TWh<extra></extra>',
)
fig2a.update_layout(title='Low-carbon energy (Nuclear + Hydro) by country, TWh',
                    margin=dict(t=60, l=10, r=10, b=10))
fig2a.show()

# --- Horizontal bar ---
top = low.iloc[0]
low_sorted = low.sort_values('TWh', ascending=True)   # biggest bar ends up on top

fig2b = px.bar(low_sorted, x='TWh', y='Country', orientation='h',
               color_discrete_sequence=['#0072B2'])
fig2b.update_traces(texttemplate='%{x:,.0f} TWh', textposition='outside',
                    cliponaxis=False)
fig2b.update_layout(
    title=f"{top['Country']} leads low-carbon output with {top['TWh']:,.0f} TWh",
    xaxis_title='Low-carbon TWh (Nuclear + Hydro)',
    yaxis_title=None,
    margin=dict(r=80),
)
fig2b.show()

### Task 2 — reflection

**Which chart tells the ranking story more clearly, and why?**

*The horizontal bar chart tells the ranking story more clearly. Bars encode value by length along a common baseline, which the eye compares very accurately, and the sorted order gives the rank directly from top to bottom. In the treemap, values are encoded by rectangle area, which is much harder to compare, and the rectangles differ in both width and height. Countries with similar totals (for example the 3rd and 4th ranked) are hard to tell apart in the treemap, while in the bar chart the small difference in bar length is visible and the TWh label confirms it. The treemap is better only for showing each country’s share of the whole at a glance, not for ranking.*

## Task 3 — 100% stacked bar: which region relies most on coal?

**What to build:** A horizontal 100% stacked bar chart showing, for each **Region**, how its fossil energy splits between Coal, Oil and Natural Gas. This is the **same data as your Task 1 treemap**, visualised a second way.

**Requirements:**
- Reuse `df_fossil` from Task 1; aggregate TWh by `Region` × `Source` and convert to **shares of each region's fossil total**
- Horizontal bars, regions **sorted by coal share** (highest at the top)
- Coal is the segment on the baseline (first in `category_orders`) and the **only coloured segment** — keep it the same colour as in Task 1; Oil and Natural Gas in two greys
- Percentage labels on the coal segments only
- Insight title naming the most coal-dependent region and its coal share

> 💡 `df.groupby('Region')['TWh'].transform('sum')` gives each row its region total

In [14]:
# Task 3
# YOUR CODE HERE
# Aggregate by Region x Source and convert to shares of each region's fossil total
reg = df_fossil.groupby(['Region', 'Source'], as_index=False)['TWh'].sum()
reg['Share'] = reg['TWh'] / reg.groupby('Region')['TWh'].transform('sum') * 100

# Region order by coal share (highest first)
coal = reg.loc[reg['Source'] == 'Coal'].sort_values('Share', ascending=False)
region_order = coal['Region'].tolist()
top_region3, top_share = coal.iloc[0]['Region'], coal.iloc[0]['Share']

# Label coal segments only
reg['label'] = np.where(reg['Source'] == 'Coal',
                        reg['Share'].round(0).astype(int).astype(str) + '%', '')

fig3 = px.bar(
    reg, x='Share', y='Region', color='Source', orientation='h', text='label',
    category_orders={'Source': ['Coal', 'Oil', 'Natural Gas'],
                     'Region': region_order},
    color_discrete_map={'Coal': '#D55E00',          # same as Task 1
                        'Oil': '#BDBDBD',
                        'Natural Gas': '#8C8C8C'},
)
fig3.update_traces(textposition='inside', insidetextanchor='middle')
fig3.update_yaxes(autorange='reversed')             # highest coal share at top
fig3.update_xaxes(range=[0, 100])
fig3.update_layout(
    title=f'{top_region3} is the most coal-dependent region: '
          f'{top_share:.0f}% of its fossil energy comes from coal',
    xaxis_title='Share of regional fossil energy (%)',
    yaxis_title=None,
    legend_title=None,
)
fig3.show()

### Task 3 — reflection

**Your Task 1 treemap and this stacked bar show the same fossil data. Name one question each chart answers that the other cannot.**

*Treemap: it answers “which country or region contributes the most fossil TWh overall?” Because area reflects absolute TWh, the biggest consumers (e.g. China and the United States) stand out immediately, and the nested hierarchy shows how each country’s total breaks down. The stacked bar cannot show this, because every region is scaled to 100% and a small region looks as large as a huge one.

Stacked bar: it answers “which region relies most heavily on coal, proportionally?” Every bar shares the same 0-100% baseline, and coal sits at the start, so coal shares can be compared directly across regions. The treemap cannot show this well, because comparing proportions across rectangles of very different sizes is hard, and a small region’s coal share is hard to read.*